# Phase 1: Continued Pre-training on Medical Literature (CPU)

**Train Qwen2.5-7B on 14 medical books using LoRA (CPU optimized)**

- **Duration:** 24-36 hours on CPU (can leave running for multiple days)
- **Memory:** 16-32 GB RAM
- **Output:** Medical-grounded Qwen model ready for Phase 2

## 1. Setup and Requirements

In [ ]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['OMP_NUM_THREADS'] = '4'  # Use 4 CPU threads
os.environ['MKL_NUM_THREADS'] = '4'

import torch
import json
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("PHASE 1: CONTINUED PRE-TRAINING (CPU)")
print("="*80)
print(f"Timestamp: {datetime.now().isoformat()}")
print(f"Device: {torch.device('cpu')}")
print(f"CPU count: {os.cpu_count()}")
print("="*80)
print()

In [ ]:
# Verify data file exists
DATA_FILE = r'C:\Users\Krish\Downloads\LLM_Finetuning\full_medical_data.txt'
OUTPUT_DIR = r'C:\Users\Krish\Downloads\LLM_Finetuning\qwen_medical_pretrained_cpu'
LORA_OUTPUT = r'C:\Users\Krish\Downloads\LLM_Finetuning\qwen_medical_lora_cpu'
CACHE_DIR = r'C:\Users\Krish\Downloads\LLM_Finetuning\.cache'

# Create directories
Path(OUTPUT_DIR).mkdir(exist_ok=True, parents=True)
Path(LORA_OUTPUT).mkdir(exist_ok=True, parents=True)
Path(CACHE_DIR).mkdir(exist_ok=True, parents=True)

if not os.path.exists(DATA_FILE):
    print(f"ERROR: Data file not found: {DATA_FILE}")
    print(f"Run extract_pdf_data.py first")
else:
    data_size_mb = os.path.getsize(DATA_FILE) / (1024 * 1024)
    print(f"[1/6] Data file verified")
    print(f"  Path: {DATA_FILE}")
    print(f"  Size: {data_size_mb:.2f} MB")
    print()
    
    # Count words
    with open(DATA_FILE, 'r', encoding='utf-8') as f:
        text = f.read()
    word_count = len(text.split())
    print(f"  Words: {word_count:,}")
    print()

## 2. Load Tokenizer and Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-7B"

print("[2/6] Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    cache_dir=CACHE_DIR
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"  ✓ Tokenizer loaded")
print(f"  Vocab size: {len(tokenizer)}")
print()

print("[3/6] Loading base model (CPU mode)...")
print("  This may take 2-5 minutes...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map='cpu',  # CPU only
    torch_dtype=torch.float32,  # CPU requires float32
    trust_remote_code=True,
    cache_dir=CACHE_DIR,
)
print(f"  ✓ Model loaded on CPU")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B")
print()

## 3. Configure LoRA

In [ ]:
from peft import get_peft_model, LoraConfig, TaskType

print("[4/6] Configuring LoRA...")

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                          # Smaller rank for CPU (less memory)
    lora_alpha=16,                # Lower alpha
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)

model = get_peft_model(model, lora_config)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())

print(f"  ✓ LoRA configured")
print(f"  LoRA rank: {lora_config.r}")
print(f"  LoRA alpha: {lora_config.lora_alpha}")
print(f"  Trainable params: {trainable_params / 1e6:.1f}M ({100 * trainable_params / all_params:.2f}%)")
print(f"  Total params: {all_params / 1e9:.1f}B")
print()

## 4. Prepare Dataset

In [ ]:
from transformers import TextDataset, DataCollatorForLanguageModeling

print("[5/6] Preparing dataset...")

train_dataset = TextDataset(
    tokenizer=tokenizer,
    file_path=DATA_FILE,
    block_size=256,  # Smaller for CPU
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

print(f"  ✓ Dataset prepared")
print(f"  Dataset size: {len(train_dataset)} blocks")
print(f"  Block size: 256 tokens")
print()

## 5. Training Configuration (CPU Optimized)

In [ ]:
from transformers import Trainer, TrainingArguments

print("[6/6] Configuring training (CPU mode)...")

# CPU-optimized settings (slower but memory efficient)
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    num_train_epochs=1,              # 1 epoch on CPU (less time)
    per_device_train_batch_size=1,   # Small batch for CPU
    gradient_accumulation_steps=4,   # Effective batch: 4
    learning_rate=5e-4,
    weight_decay=0.01,
    warmup_steps=100,
    logging_steps=20,                # Log every 20 steps (for CPU)
    save_strategy="steps",
    save_steps=200,                  # Save checkpoint every 200 steps
    save_total_limit=2,              # Keep last 2 checkpoints
    load_best_model_at_end=False,    # Skip for CPU (slow)
    fp16=False,                      # CPU doesn't support fp16
    gradient_checkpointing=False,    # Not beneficial on CPU
    max_grad_norm=1.0,
    lr_scheduler_type="linear",
    log_level="info",
    report_to=[],                    # No tensorboard on CPU
    dataloader_num_workers=0,        # CPU training
    remove_unused_columns=False,
)

print(f"  ✓ Training configuration ready")
print(f"  Training mode: CPU")
print(f"  Batch size (effective): {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Estimated time: 24-36 hours")
print()

## 6. Start Training

⚠️ **WARNING: This will take 24-36 hours on CPU. You can leave it running overnight/for multiple days.**

Training will save checkpoints every 200 steps, so you won't lose progress if interrupted.

In [ ]:
print("Starting training...")
print(f"Time: {datetime.now().isoformat()}")
print()

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
)

train_result = trainer.train()

print()
print("="*80)
print("TRAINING COMPLETE")
print("="*80)
print(f"Training loss: {train_result.training_loss:.4f}")
print(f"Time: {datetime.now().isoformat()}")
print()

## 7. Save LoRA Adapters

In [ ]:
print("Saving LoRA adapters...")

model.save_pretrained(LORA_OUTPUT)
tokenizer.save_pretrained(LORA_OUTPUT)

# Save config
config = {
    'model': MODEL_NAME,
    'device': 'CPU',
    'data_file': DATA_FILE,
    'lora_rank': lora_config.r,
    'lora_alpha': lora_config.lora_alpha,
    'training_epochs': training_args.num_train_epochs,
    'training_loss': float(train_result.training_loss),
    'timestamp': datetime.now().isoformat(),
}

with open(os.path.join(LORA_OUTPUT, 'training_config.json'), 'w') as f:
    json.dump(config, f, indent=2)

print(f"✓ LoRA saved to: {LORA_OUTPUT}")
print(f"✓ Config saved")
print()
print("="*80)
print("PHASE 1 COMPLETE")
print("="*80)
print()
print("Next: Phase 2 - Instruction Fine-tuning")
print("  Run: Phase2_Instruction_Finetuning_CPU.ipynb")

## Notes

- **Total time:** 24-36 hours on CPU
- **Checkpoints:** Saved every 200 steps to `qwen_medical_lora_cpu/checkpoint-*`
- **Resume training:** If interrupted, run the training cell again - it will resume from the last checkpoint
- **Memory usage:** ~16-32 GB RAM (depending on block size)

If memory runs out:
- Reduce `per_device_train_batch_size` to 1 in training_args
- Reduce `block_size` to 128 in dataset preparation